# 09 — Notification Text Bake-Off

**Goal:** Find the cheapest text model that writes weather-aware outfit notification messages as well as `gemini-2.5-flash` — the current model used for reminder text.

**Pipeline:**
```
3 fixed (weather + outfit + occasion) prompts
  → each model generates a notification message
  → side-by-side grid for eyeball scoring
  → latency + actual billed cost table
  → decision: which model to swap into the notification cron
```


In [ ]:
%matplotlib inline
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str(Path("..")))

from lib.providers import GeminiProvider, OpenRouterProvider, all_providers, print_availability

OUT_DIR = Path("../outputs/09")
OUT_DIR.mkdir(parents=True, exist_ok=True)

providers = all_providers()
print("Provider status:")
print_availability(providers)


In [ ]:
# ── Fixed prompts ─────────────────────────────────────────────────────────────
PROMPTS = [{'id': 'rainy_morning', 'system': 'You write short, friendly daily style messages for a personal wardrobe app. Be warm, concise (2-3 sentences max), and specific to the clothes mentioned.', 'user': "Weather: 13°C, rainy, humidity 85%. User's city: London. Outfit: navy trench coat + white Oxford shirt + dark slim jeans + tan Chelsea boots. Occasion: casual work day. Write a morning notification message."}, {'id': 'hot_weekend', 'system': 'You write short, friendly daily style messages for a personal wardrobe app. Be warm, concise (2-3 sentences max), and specific to the clothes mentioned.', 'user': "Weather: 32°C, sunny, UV index 8. User's city: Dubai. Outfit: white linen shirt + khaki shorts + white canvas sneakers. Occasion: weekend brunch. Write a morning notification message."}, {'id': 'cold_formal', 'system': 'You write short, friendly daily style messages for a personal wardrobe app. Be warm, concise (2-3 sentences max), and specific to the clothes mentioned.', 'user': "Weather: -2°C, light snow. User's city: Toronto. Outfit: charcoal wool suit + black turtleneck + black leather oxfords. Occasion: job interview. Write a morning notification message."}]

for p in PROMPTS:
    print(f"[{p['id']}]  {p['user'][:80]}...")


In [ ]:
# ── Models ────────────────────────────────────────────────────────────────────
MODELS = [
    ("gemini",      "gemini-2.5-flash"),
    ("openrouter",  "deepseek/deepseek-chat-v3.1"),
    ("openrouter",  "qwen/qwen3-30b-a3b-instruct-2507"),
    ("openrouter",  "qwen/qwen3-8b"),
    ("openrouter",  "z-ai/glm-4.5-air"),
    ("openrouter",  "moonshotai/kimi-k2"),
    ("openrouter",  "mistralai/mistral-small-3.2-24b-instruct"),
]

print("Models to compare:")
for pname, mid in MODELS:
    p = providers[pname]
    status = "ready" if p.available else "SKIP (key missing)"
    print(f"  [{status}] {mid}")


In [ ]:
# ── Run all models on all prompts ─────────────────────────────────────────────
results = {}  # model_id → list[TextResult]

for provider_name, model_id in MODELS:
    p = providers[provider_name]
    if not p.available:
        print(f"SKIP {model_id}")
        continue
    print(f"\n[{model_id}]")
    model_results = []
    for prompt in PROMPTS:
        messages = [
            {"role": "system", "content": prompt["system"]},
            {"role": "user",   "content": prompt["user"]},
        ]
        r = p.chat_text(model_id, messages)
        tag = "✓" if r.ok else "✗"
        cost_str = f"${r.cost_usd:.6f}{'*' if r.cost_is_actual else '~'}"
        routed = f" via {r.routed_provider}" if r.routed_provider else ""
        print(f"  [{prompt['id']}]: {tag} {r.latency_s:.1f}s {cost_str}{routed}")
        if r.ok:
            print(f"    → {r.text[:120].strip()}")
        else:
            print(f"    ERR: {r.error}")
        model_results.append(r)
    results[model_id] = model_results

print("\n* = actual billed   ~ = estimate")


In [ ]:
# ── Side-by-side grid: rows = prompts, columns = models ──────────────────────
model_ids = list(results.keys())
n_models  = len(model_ids)
n_prompts = len(PROMPTS)

if n_models == 0:
    print("No results")
else:
    fig, axes = plt.subplots(n_prompts, n_models, figsize=(n_models * 4, n_prompts * 3.5))
    if n_prompts == 1: axes = [axes]
    if n_models == 1:  axes = [[ax] for ax in axes]

    for row, prompt in enumerate(PROMPTS):
        for col, model_id in enumerate(model_ids):
            ax = axes[row][col]
            r = results[model_id][row]
            text = r.text.strip() if r.ok else f"ERROR:\n{r.error}"
            cost = f"${r.cost_usd:.6f}{'*' if r.cost_is_actual else '~'} | {r.latency_s:.1f}s"
            ax.text(0.05, 0.95, text, transform=ax.transAxes,
                    fontsize=8, verticalalignment="top", wrap=True,
                    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.7))
            ax.text(0.05, 0.05, cost, transform=ax.transAxes,
                    fontsize=6.5, color="gray")
            if row == 0:
                ax.set_title(model_id.split("/")[-1][:22], fontsize=8)
            if col == 0:
                ax.set_ylabel(prompt["id"], fontsize=8)
            ax.axis("off")

    plt.suptitle("Notification text — all prompts × all models", fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# ── Metrics table ─────────────────────────────────────────────────────────────
rows = []
for model_id, model_results in results.items():
    total_cost  = sum(r.cost_usd for r in model_results)
    avg_latency = sum(r.latency_s for r in model_results) / len(model_results) if model_results else 0
    total_tok_in  = sum(r.tokens_in  for r in model_results)
    total_tok_out = sum(r.tokens_out for r in model_results)
    cost_actual = all(r.cost_is_actual for r in model_results if r.ok)
    routed = model_results[0].routed_provider if model_results else ""
    rows.append({
        "Model": model_id.split("/")[-1][:28],
        "Routed via": routed or "—",
        "Avg latency (s)": f"{avg_latency:.1f}",
        "Total cost (3 prompts)": f"${total_cost:.6f}{'*' if cost_actual else '~'}",
        "Tokens in / out": f"{total_tok_in} / {total_tok_out}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(OUT_DIR / "09_metrics.csv", index=False)
print("\n* = actual billed   ~ = estimate")


In [ ]:
# ── Vibes scoring template ────────────────────────────────────────────────────
# Fill in scores 1-5 after reviewing the grid above.
# Tone: natural & friendly | Brevity: ≤3 sentences | Weather-aware: references temp/conditions
# Outfit-specific: names the garment | Overall: would you want this as your morning notification?

vibes = {
    "gemini-2.5-flash":                          {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "deepseek-chat-v3.1":                        {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "qwen3-30b-a3b-instruct-2507":               {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "qwen3-8b":                                  {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "glm-4.5-air":                               {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "kimi-k2":                                   {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
    "mistral-small-3.2-24b-instruct":            {"Tone": 0, "Brevity": 0, "Weather": 0, "Outfit": 0, "Overall": 0},
}
# Edit the 0s above after reading the outputs, then re-run this cell.
print(pd.DataFrame(vibes).T.to_string())


## Decision note

Fill in after reviewing outputs and vibes scores above.

**Winner:** `_______`

**Why:**
- Vibes overall score: 
- Cost per notification vs Gemini: 
- Latency: 
- Routed via: 

**Ship recommendation for the notification cron:**
```
provider: openrouter
model:    _______
```

**Open questions:**
- Does the winner stay concise with longer closet descriptions?
- Does adding few-shot examples narrow the tone gap with Gemini?
